In [ ]:
"""
Imports variables from urban_data_variables module.
Replaces 'obns_lat' from district_codes with the correct latin spelling.
"""

import pandas as pd
import geopandas as gpd
import re
from urban_data_variables import (
    districts,
    district_codes,
    schools,
    kindergartens,
    parking_spaces,
    public_transport_accessibility,
    parks_accessibility,
    healthcare_coverage
)


In [21]:
import cyrtranslit

# Applies a lambda function to latinize district names
district_codes['obns_lat'] = district_codes['obns_cyr'].apply(lambda x: cyrtranslit.to_latin(x, 'bg'))
# Removes diacritical A
district_codes['obns_lat'] = district_codes['obns_lat'].apply(lambda x: x.replace('Ă', 'A'))


In [ ]:
neighborhoods_list = districts['regname'].tolist()
neighborhoods_list = [cyrtranslit.to_latin(n, 'bg') for n in neighborhoods_list]
neighborhoods_list = [n.replace('Ă', 'A').replace('ă', 'a') for n in neighborhoods_list]

# Split several names into different entries
neighborhoods_list = sorted({part.strip() for n in neighborhoods_list for part in re.split(r',', n) if part.strip()})

# Remove small words
neighborhoods_list = [re.sub(r'\b\w{1,2}\.', '', n) for n in neighborhoods_list]
neighborhoods_list = [n.strip() for n in neighborhoods_list]

def tidy(n):
    n = re.sub(r'\s*-\s*(\d+)\s*mr\b', r' \1', n, flags=re.IGNORECASE)  # "Lyulin - 10 mr" -> "Lyulin 10"
    return re.sub(r'\s+', ' ', n).strip()

neighborhoods_list = sorted({tidy(n) for n in neighborhoods_list})

neighborhoods_list

In [ ]:
"""
Mapping districts and neighborhoods to the scraped listings files
"""

sofia_for_rent_scraped = pd.read_csv(r'D:\Data Analysis\5_Projects\sofia_housing_districts_analysis\data\housing_listings_scraped\sofia_for_rent_scraped.csv')

districts_list = district_codes['obns_lat'].tolist()

pattern_districts = '|'.join(re.escape(d) for d in sorted(districts_list, key=len, reverse=True))
pattern_neighborhoods = '|'.join(re.escape(d) for d in sorted(neighborhoods_list, key=len, reverse=True))

sofia_for_rent_scraped['district'] = sofia_for_rent_scraped['location'].str.extract(rf'\b({pattern_districts})\b', flags=re.IGNORECASE)[0]
sofia_for_rent_scraped['neighborhood'] = sofia_for_rent_scraped['location'].str.extract(rf'\b({pattern_neighborhoods})\b', flags=re.IGNORECASE)[0]

# match rate
len(sofia_for_rent_scraped[(sofia_for_rent_scraped['neighborhood'].notna()) | (sofia_for_rent_scraped['district'].notna())]) / len(sofia_for_rent_scraped)

0.5711145096539858

In [110]:
sofia_for_sale_scraped = pd.read_csv(r'D:\Data Analysis\5_Projects\sofia_housing_districts_analysis\data\housing_listings_scraped\sofia_for_sale_scraped.csv')

sofia_for_sale_scraped['district'] = sofia_for_sale_scraped['location'].str.extract(rf'\b({pattern_districts})\b', flags=re.IGNORECASE)[0]
sofia_for_sale_scraped['neighborhood'] = sofia_for_sale_scraped['location'].str.extract(rf'\b({pattern_neighborhoods})\b', flags=re.IGNORECASE)[0]

# match rate
len(sofia_for_sale_scraped[(sofia_for_rent_scraped['neighborhood'].notna()) | (sofia_for_sale_scraped['district'].notna())]) / len(sofia_for_rent_scraped)

0.6298030969221946